In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_customer_portfolio
#
# Layer
# -----
# Gold Layer - Data Products
#
# Purpose
# -------
# Build a Customer Portfolio data product that consolidates
# customer profile, exposure, expected credit loss, regulatory
# indicators and executive risk flags.
#
# Grain
# -----
# One row per customer.
#
# Output
# ------
# gold_customer_portfolio
#
# Enterprise Concepts
# -------------------
# ✓ Customer 360
# ✓ Portfolio Analytics
# ✓ Executive Reporting
# ✓ Concentration Risk
# ✓ AI-ready Analytical Dataset
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

customer_table = "dim_customer"
country_table = "dim_country"
industry_table = "dim_industry"
loan_fact_table = "fact_loan_exposure"
ecl_fact_table = "fact_expected_credit_loss"

target_table = "gold_customer_portfolio"
pipeline_name = "nb_build_customer_portfolio"

run_start_time = datetime.now()

print("ERIP Customer Portfolio Build Started")

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 3, Finished, Available, Finished, False)

ERIP Customer Portfolio Build Started


In [2]:
# ============================================================
# SECTION 2 - READ GOLD TABLES
# ============================================================

dim_customer = spark.table(customer_table)
dim_country = spark.table(country_table)
dim_industry = spark.table(industry_table)
fact_loan_exposure = spark.table(loan_fact_table)
fact_expected_credit_loss = spark.table(ecl_fact_table)

print(f"Customers : {dim_customer.count()}")
print(f"Countries : {dim_country.count()}")
print(f"Industries: {dim_industry.count()}")
print(f"Loans     : {fact_loan_exposure.count()}")
print(f"ECL Rows  : {fact_expected_credit_loss.count()}")

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 4, Finished, Available, Finished, False)

Customers : 1000
Countries : 6
Industries: 10
Loans     : 5000
ECL Rows  : 5000


In [3]:
# ============================================================
# SECTION 3 - CUSTOMER LOAN AGGREGATION
# ============================================================
#
# Purpose
# -------
# Aggregate loan exposure and Basel-style RWA measures at
# customer grain.
# ============================================================

loan_summary = (
    fact_loan_exposure
    .groupBy("customer_sk")
    .agg(
        countDistinct("loan_id").alias("number_of_loans"),
        sum("approved_limit").alias("total_approved_limit"),
        sum("outstanding_balance").alias("total_outstanding_balance"),
        sum("exposure_at_default").alias("total_exposure"),
        sum("risk_weighted_assets").alias("total_rwa"),
        avg("interest_rate_pct").alias("average_interest_rate"),
        avg("utilization_pct").alias("average_utilization")
    )
)

print(f"Loan summary rows: {loan_summary.count()}")
display(loan_summary.limit(10))

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 5, Finished, Available, Finished, False)

Loan summary rows: 995


SynapseWidget(Synapse.DataFrame, 88e4b38e-6fad-460b-8180-ee06615886b8)

In [4]:
# ============================================================
# SECTION 4 - CUSTOMER EXPECTED CREDIT LOSS AGGREGATION
# ============================================================
#
# Purpose
# -------
# Aggregate IFRS 9 Expected Credit Loss and rating metrics
# at customer grain.
# ============================================================

ecl_summary = (
    fact_expected_credit_loss
    .groupBy("customer_sk")
    .agg(
        sum("calculated_ecl").alias("total_expected_credit_loss"),
        avg("pd").alias("average_pd"),
        avg("lgd").alias("average_lgd"),
        max("ifrs9_stage_numeric").alias("highest_ifrs_stage"),
        max("current_internal_grade").alias("current_internal_grade")
    )
)

print(f"ECL summary rows: {ecl_summary.count()}")
display(ecl_summary.limit(10))

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 6, Finished, Available, Finished, False)

ECL summary rows: 995


SynapseWidget(Synapse.DataFrame, 117db558-c110-474c-ae56-5f6b4119a486)

In [5]:
# ============================================================
# SECTION 5 - BUILD CUSTOMER PORTFOLIO DATA PRODUCT
# ============================================================
#
# Purpose
# -------
# Combine customer dimension attributes with exposure and ECL
# measures to create a business-ready data product.
# ============================================================

gold_customer_portfolio = (
    dim_customer.alias("c")
    .join(loan_summary.alias("l"), "customer_sk", "left")
    .join(ecl_summary.alias("e"), "customer_sk", "left")
    .join(dim_country.alias("co"), "country_code", "left")
    .join(dim_industry.alias("i"), "industry_code", "left")
    .fillna({
        "number_of_loans": 0,
        "total_approved_limit": 0,
        "total_outstanding_balance": 0,
        "total_exposure": 0,
        "total_rwa": 0,
        "total_expected_credit_loss": 0,
        "average_pd": 0,
        "average_lgd": 0,
        "average_interest_rate": 0,
        "average_utilization": 0
    })
    .withColumn(
        "portfolio_risk_level",
        when(col("total_expected_credit_loss") >= 5000000, "Critical")
        .when(col("total_expected_credit_loss") >= 1000000, "High")
        .when(col("total_expected_credit_loss") >= 250000, "Medium")
        .otherwise("Low")
    )
    .withColumn(
        "strategic_customer_flag",
        when(col("total_exposure") >= 50000000, "Y").otherwise("N")
    )
    .withColumn(
        "high_risk_flag",
        when(
            (col("portfolio_risk_level").isin("Critical", "High")) |
            (col("customer_risk_category") == "High Risk") |
            (col("highest_ifrs_stage") >= 2),
            "Y"
        ).otherwise("N")
    )
)

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 7, Finished, Available, Finished, False)

In [7]:
# ============================================================
# SECTION 6 - EXECUTIVE RANKING AND CONCENTRATION METRICS
# ============================================================
#
# Purpose
# -------
# Add executive-friendly indicators used for concentration
# risk, prioritization, Power BI drill-down and AI summaries.
#
# Derived Metrics
# ---------------
# • Exposure rank
# • ECL rank
# • RWA rank
# • Exposure concentration %
# • ECL concentration %
# ============================================================

total_portfolio_exposure = (
    gold_customer_portfolio
    .agg(sum("total_exposure").alias("total"))
    .collect()[0]["total"]
)

total_portfolio_ecl = (
    gold_customer_portfolio
    .agg(sum("total_expected_credit_loss").alias("total"))
    .collect()[0]["total"]
)

exposure_rank_window = Window.orderBy(col("total_exposure").desc())
ecl_rank_window = Window.orderBy(col("total_expected_credit_loss").desc())
rwa_rank_window = Window.orderBy(col("total_rwa").desc())

gold_customer_portfolio = (
    gold_customer_portfolio
    .withColumn("exposure_rank", row_number().over(exposure_rank_window))
    .withColumn("ecl_rank", row_number().over(ecl_rank_window))
    .withColumn("rwa_rank", row_number().over(rwa_rank_window))
    .withColumn(
        "exposure_concentration_pct",
        when(
            lit(total_portfolio_exposure) > 0,
            (col("total_exposure") / lit(total_portfolio_exposure)) * 100
        ).otherwise(0)
    )
    .withColumn(
        "ecl_concentration_pct",
        when(
            lit(total_portfolio_ecl) > 0,
            (col("total_expected_credit_loss") / lit(total_portfolio_ecl)) * 100
        ).otherwise(0)
    )
    .select(

    col("c.customer_sk"),
    col("c.customer_id"),
    col("c.customer_name"),
    col("c.customer_group_id"),
    col("c.segment"),

    col("co.country").alias("country"),
    col("co.region").alias("region"),

    col("i.industry_name").alias("industry_name"),

    col("c.customer_risk_category"),
    col("c.kyc_risk_rating"),
    col("c.esg_score"),

    col("number_of_loans"),
    col("total_approved_limit"),
    col("total_outstanding_balance"),
    col("total_exposure"),
    col("total_rwa"),
    col("total_expected_credit_loss"),
    col("average_pd"),
    col("average_lgd"),
    col("highest_ifrs_stage"),
    col("current_internal_grade"),
    col("average_interest_rate"),
    col("average_utilization"),
    col("portfolio_risk_level"),
    col("strategic_customer_flag"),
    col("high_risk_flag"),
    col("exposure_rank"),
    col("ecl_rank"),
    col("rwa_rank"),
    col("exposure_concentration_pct"),
    col("ecl_concentration_pct"),

    current_timestamp().alias("gold_updated_timestamp")
)
)

print(f"Rows created: {gold_customer_portfolio.count()}")
display(gold_customer_portfolio.limit(10))

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 9, Finished, Available, Finished, False)

Rows created: 1000


SynapseWidget(Synapse.DataFrame, 37d29508-eee8-4ca5-bff9-b0403310ad40)

In [8]:
# ============================================================
# SECTION 7 - DATA PRODUCT QUALITY CHECKS
# ============================================================

total_rows = gold_customer_portfolio.count()

duplicate_customers = (
    total_rows -
    gold_customer_portfolio.select("customer_id").distinct().count()
)

null_customer_sk = gold_customer_portfolio.filter(
    col("customer_sk").isNull()
).count()

null_customer_name = gold_customer_portfolio.filter(
    col("customer_name").isNull()
).count()

negative_exposure = gold_customer_portfolio.filter(
    col("total_exposure") < 0
).count()

negative_ecl = gold_customer_portfolio.filter(
    col("total_expected_credit_loss") < 0
).count()

print("Customer Portfolio Quality Checks")
print("---------------------------------")
print(f"Rows                 : {total_rows}")
print(f"Duplicate Customers  : {duplicate_customers}")
print(f"Null Customer SK     : {null_customer_sk}")
print(f"Null Customer Name   : {null_customer_name}")
print(f"Negative Exposure    : {negative_exposure}")
print(f"Negative ECL         : {negative_ecl}")

if (
    duplicate_customers > 0 or
    null_customer_sk > 0 or
    null_customer_name > 0 or
    negative_exposure > 0 or
    negative_ecl > 0
):
    raise Exception("Customer Portfolio Validation Failed")
else:
    print("✓ Customer Portfolio Validation Passed")

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 10, Finished, Available, Finished, False)

Customer Portfolio Quality Checks
---------------------------------
Rows                 : 1000
Duplicate Customers  : 0
Null Customer SK     : 0
Null Customer Name   : 0
Negative Exposure    : 0
Negative ECL         : 0
✓ Customer Portfolio Validation Passed


In [9]:
# ============================================================
# SECTION 8 - WRITE GOLD CUSTOMER PORTFOLIO DATA PRODUCT
# ============================================================

(
    gold_customer_portfolio.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(target_table)
)

print(f"✓ Gold data product created: {target_table}")
print(f"Rows written: {gold_customer_portfolio.count()}")

StatementMeta(, 93b0d941-861d-48c8-b2c7-a6a278129366, 11, Finished, Available, Finished, False)

✓ Gold data product created: gold_customer_portfolio
Rows written: 1000
